# S² Rational Stability Thresholds

This notebook verifies the **exact rational stability thresholds** for the regular $N$-polygon of point vortices on the 2-sphere $S^2$.

On $S^2$, curvature *destabilizes* the ring (unlike $\mathbb{H}^2$ where curvature stabilizes). The key result is that the thresholds $\xi_{\mathrm{crit}}$ are all **rational numbers**, and only $N \leq 6$ admit a stable regime.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from fractions import Fraction
import math

## The $C_1$ Formula on $S^2$

On $S^2$ with Gaussian curvature $K = 1/R^2$, the curvature-corrected Havelock coefficient for the $N$-ring at colatitude $\varphi_0$ is:

$$C_1(S^2, \xi) = (N-1)\frac{1 - \xi}{1 + \xi} = (N-1)\cos\varphi_0$$

where $\xi = \tan^2(\varphi_0/2)$ is the stereographic radius squared. The identity connecting these two forms follows from the half-angle substitution:

$$\cos\varphi_0 = \frac{1 - \tan^2(\varphi_0/2)}{1 + \tan^2(\varphi_0/2)} = \frac{1 - \xi}{1 + \xi}$$

Key properties:
- At the pole ($\xi = 0$, $\varphi_0 = 0$): $C_1 = N-1$ (flat-plane limit)
- At the equator ($\xi = 1$, $\varphi_0 = \pi/2$): $C_1 = 0$ (marginal)
- $C_1$ is **monotonically decreasing** in $\xi$, so curvature destabilizes

The ring becomes unstable when $C_1 < \lambda_m^{\text{min}} = m(N-m)/2$ where $m = \lfloor N/2 \rfloor$.

In [ ]:
def C1_S2(N, xi):
    """Curvature coefficient on S^2."""
    return (N - 1) * (1 - xi) / (1 + xi)

# Verify the cos(phi) identity for several colatitudes
print("Verifying C1(S^2, xi) = (N-1)*cos(phi_0):")
print(f"{'phi_0 (deg)':>12} {'xi':>12} {'C1(xi)':>12} {'(N-1)*cos':>12} {'match':>8}")
print("-" * 60)

N = 6
for phi_deg in [0, 15, 30, 45, 60, 75, 90]:
    phi = np.radians(phi_deg)
    xi = np.tan(phi / 2) ** 2
    c1_from_xi = C1_S2(N, xi)
    c1_from_cos = (N - 1) * np.cos(phi)
    match = np.isclose(c1_from_xi, c1_from_cos)
    print(f"{phi_deg:>12.1f} {xi:>12.6f} {c1_from_xi:>12.6f} {c1_from_cos:>12.6f} {str(match):>8}")

## Exact Rational Thresholds

The marginal stability condition $C_1(S^2, \xi_{\mathrm{crit}}) = m(N-m)/2$ gives:

$$(N-1)\frac{1 - \xi_{\mathrm{crit}}}{1 + \xi_{\mathrm{crit}}} = \frac{m(N-m)}{2}$$

Solving for $\xi_{\mathrm{crit}}$:

$$\xi_{\mathrm{crit}} = \frac{(N-1) - m(N-m)/2}{(N-1) + m(N-m)/2}$$

Since both numerator and denominator are rational (in fact, integers or half-integers), $\xi_{\mathrm{crit}}$ is always **exactly rational**. This is in sharp contrast to the $\mathbb{H}^2$ thresholds, which generically live in quadratic extensions of $\mathbb{Q}$.

Expected values: $N=3 \to 1/3$, $N=4 \to 1/5$, $N=5 \to 1/7$, $N=6 \to 1/19$, $N \geq 7 \to$ None.

In [ ]:
from planetary_polygons.extensions.algebraic_thresholds import sphere_stability_threshold

# Known exact thresholds
expected = {
    3: Fraction(1, 3),
    4: Fraction(1, 5),
    5: Fraction(1, 7),
    6: Fraction(1, 19),
}

print("Exact S^2 stability thresholds")
print(f"{'N':>3}  {'xi_crit':>10}  {'expected':>10}  {'match':>6}")
print("-" * 38)

for N in range(3, 7):
    xi = sphere_stability_threshold(N)
    exp = expected[N]
    match = xi == exp
    print(f"{N:>3}  {str(xi):>10}  {str(exp):>10}  {str(match):>6}")
    assert match, f"N={N}: got {xi}, expected {exp}"

print()
print("N >= 7 (should return None):")
for N in range(7, 11):
    xi = sphere_stability_threshold(N)
    print(f"  N={N}: xi_crit = {xi}")
    assert xi is None, f"N={N}: expected None, got {xi}"

print("\nAll thresholds verified.")

In [ ]:
# Full table with algebraic details
print("Complete S^2 threshold table")
print(f"{'N':>3}  {'m':>3}  {'m(N-m)/2':>10}  {'N-1':>5}  {'numer':>6}  {'denom':>6}  {'xi_crit':>10}")
print("-" * 55)

for N in range(3, 11):
    m = N // 2
    T = Fraction(m * (N - m), 2)
    numer = Fraction(N - 1) - T
    denom_val = Fraction(N - 1) + T
    xi = sphere_stability_threshold(N)
    xi_str = str(xi) if xi is not None else "None"
    print(f"{N:>3}  {m:>3}  {str(T):>10}  {N-1:>5}  {str(numer):>6}  {str(denom_val):>6}  {xi_str:>10}")

## Colatitude Values

Each rational threshold $\xi_{\mathrm{crit}}$ corresponds to a critical colatitude $\varphi_{\mathrm{crit}}$ via:

$$\varphi_{\mathrm{crit}} = 2\arctan\sqrt{\xi_{\mathrm{crit}}} = \arccos\frac{1 - \xi_{\mathrm{crit}}}{1 + \xi_{\mathrm{crit}}}$$

The ring is stable for $\varphi_0 < \varphi_{\mathrm{crit}}$ (closer to the pole) and unstable beyond.

Expected values: $N=3 \to 60.0^\circ$, $N=4 \to 48.2^\circ$, $N=5 \to 41.4^\circ$, $N=6 \to 25.8^\circ$.

In [ ]:
# Expected colatitudes in degrees (to 1 decimal place)
expected_phi_deg = {3: 60.0, 4: 48.2, 5: 41.4, 6: 25.8}

print("Critical colatitudes on S^2")
print(f"{'N':>3}  {'xi_crit':>10}  {'phi (atan)':>12}  {'phi (acos)':>12}  {'expected':>10}  {'match':>6}")
print("-" * 62)

for N in range(3, 7):
    xi = sphere_stability_threshold(N)
    xi_float = float(xi)
    
    # Method 1: phi = 2*arctan(sqrt(xi))
    phi_atan = 2 * np.arctan(np.sqrt(xi_float))
    phi_atan_deg = np.degrees(phi_atan)
    
    # Method 2: phi = arccos((1-xi)/(1+xi))
    phi_acos = np.arccos((1 - xi_float) / (1 + xi_float))
    phi_acos_deg = np.degrees(phi_acos)
    
    # Verify the two methods agree
    assert np.isclose(phi_atan_deg, phi_acos_deg), \
        f"N={N}: atan={phi_atan_deg:.4f} != acos={phi_acos_deg:.4f}"
    
    # Check against expected (to 1 decimal place)
    match = abs(phi_atan_deg - expected_phi_deg[N]) < 0.1
    
    print(f"{N:>3}  {str(xi):>10}  {phi_atan_deg:>12.4f}  {phi_acos_deg:>12.4f}  "
          f"{expected_phi_deg[N]:>10.1f}  {str(match):>6}")
    assert match, f"N={N}: phi={phi_atan_deg:.4f}, expected ~{expected_phi_deg[N]}"

print("\nAll colatitudes verified.")

In [ ]:
# Also verify at the critical point: C1 exactly equals m(N-m)/2
print("Verification: C1(S^2, xi_crit) = m(N-m)/2 (exact rational arithmetic)")
print(f"{'N':>3}  {'xi_crit':>8}  {'C1(xi_crit)':>14}  {'m(N-m)/2':>10}  {'exact':>6}")
print("-" * 48)

for N in range(3, 7):
    xi = sphere_stability_threshold(N)
    m = N // 2
    T_half = Fraction(m * (N - m), 2)
    C1 = Fraction(N - 1) * (1 - xi) / (1 + xi)
    exact = (C1 == T_half)
    print(f"{N:>3}  {str(xi):>8}  {str(C1):>14}  {str(T_half):>10}  {str(exact):>6}")
    assert exact, f"N={N}: C1={C1} != T_half={T_half}"

print("\nAll marginal conditions satisfied exactly.")

## Pattern $1/(2N-3)$ and Its Break at $N = 6$

For small $N$, the thresholds follow a suggestive pattern:

| $N$ | $\xi_{\mathrm{crit}}$ | $1/(2N-3)$ | match? |
|-----|----------------------|-------------|--------|
| 3   | 1/3                  | 1/3         | yes    |
| 4   | 1/5                  | 1/5         | yes    |
| 5   | 1/7                  | 1/7         | yes    |
| 6   | **1/19**             | 1/9         | **no** |

**Why does the pattern break at $N = 6$?**

The formula $\xi_{\mathrm{crit}} = 1/(2N-3)$ holds iff $m(N-m)/2 = N-2$, which requires $m(N-m) = 2(N-2)$.

For odd $N$, $m = (N-1)/2$, so $m(N-m) = (N-1)(N+1)/4$. The condition $(N^2-1)/4 = 2(N-2)$ gives $N^2 - 8N + 15 = 0$, i.e. $N = 3$ or $N = 5$.

For even $N$, $m = N/2$, so $m(N-m) = N^2/4$. The condition $N^2/4 = 2(N-2)$ gives $N^2 - 8N + 16 = 0$, i.e. $(N-4)^2 = 0$, so $N = 4$ only.

Thus $N \in \{3, 4, 5\}$ are the **only** cases where $\xi_{\mathrm{crit}} = 1/(2N-3)$. At $N = 6$, $m(N-m)/2 = 3 \cdot 3 = 9/1$, while $N - 2 = 4$, so the simple pattern fails and we get $\xi_{\mathrm{crit}} = 1/19$ instead of $1/9$.

In [ ]:
# Verify the 1/(2N-3) pattern and its breakdown
print("Pattern analysis: xi_crit vs 1/(2N-3)")
print(f"{'N':>3}  {'m':>3}  {'m(N-m)/2':>10}  {'N-2':>5}  {'equal?':>7}  {'xi_crit':>10}  {'1/(2N-3)':>10}  {'match':>6}")
print("-" * 68)

for N in range(3, 9):
    m = N // 2
    T = Fraction(m * (N - m), 2)
    nm2 = N - 2
    T_eq_nm2 = (T == nm2)
    
    xi = sphere_stability_threshold(N)
    simple = Fraction(1, 2 * N - 3) if (2 * N - 3) > 0 else None
    
    xi_str = str(xi) if xi is not None else "None"
    simple_str = str(simple) if simple is not None else "N/A"
    match = xi == simple if xi is not None else False
    
    print(f"{N:>3}  {m:>3}  {str(T):>10}  {nm2:>5}  {str(T_eq_nm2):>7}  {xi_str:>10}  {simple_str:>10}  {str(match):>6}")

In [ ]:
# Solve the condition m(N-m) = 2(N-2) algebraically
print("Algebraic verification: for which N does m(N-m)/2 = N-2?")
print()

# Odd N: m = (N-1)/2, so m(N-m) = (N-1)(N+1)/4 = (N^2-1)/4
# Condition: (N^2-1)/4 = 2(N-2) => N^2-1 = 8N-16 => N^2-8N+15 = 0
# => (N-3)(N-5) = 0 => N = 3 or N = 5
print("Odd N:  N^2 - 8N + 15 = 0  =>  (N-3)(N-5) = 0  =>  N = 3 or N = 5")
for N in [3, 5]:
    val = N**2 - 8*N + 15
    print(f"  N={N}: {N}^2 - 8*{N} + 15 = {val}")

print()

# Even N: m = N/2, so m(N-m) = N^2/4
# Condition: N^2/4 = 2(N-2) => N^2 = 8N-16 => N^2-8N+16 = 0
# => (N-4)^2 = 0 => N = 4 (double root)
print("Even N: N^2 - 8N + 16 = 0  =>  (N-4)^2 = 0  =>  N = 4 only")
for N in [4, 6]:
    val = N**2 - 8*N + 16
    print(f"  N={N}: {N}^2 - 8*{N} + 16 = {val}")

print()
print("Conclusion: only N in {3, 4, 5} satisfy xi_crit = 1/(2N-3).")
print("At N=6, m(N-m)/2 = 9/2 != 4 = N-2, giving xi_crit = 1/19 instead of 1/9.")

In [ ]:
# Explicit check: N=6 has m(N-m)/2 = 9/2, not N-2 = 4
N = 6
m = N // 2  # = 3
T = Fraction(m * (N - m), 2)  # = 3*3/2 = 9/2

print(f"N = {N}, m = {m}")
print(f"m(N-m)/2 = {m}*{N-m}/2 = {T}")
print(f"N - 2    = {N - 2}")
print(f"Equal?     {T == N - 2}")
print()

# What 1/(2N-3) would give:
xi_simple = Fraction(1, 2*N - 3)
# What the actual threshold is:
xi_actual = sphere_stability_threshold(N)

print(f"Naive 1/(2N-3)  = {xi_simple} = 1/{2*N-3}")
print(f"Actual xi_crit  = {xi_actual}")
print(f"Ratio actual/naive = {float(xi_actual)/float(xi_simple):.6f}")

## Even/Odd Closed Forms

The module documents explicit closed-form expressions for the S^2 thresholds:

- **Even $N$**: $\xi_{\mathrm{crit}} = \frac{8N - 8 - N^2}{N^2 + 8N - 8}$
- **Odd $N$**: $\xi_{\mathrm{crit}} = \frac{7 - N}{N + 9}$

Let us verify these against the general formula.

In [ ]:
print("Even/odd closed-form verification")
print(f"{'N':>3}  {'parity':>6}  {'closed-form':>14}  {'general':>10}  {'match':>6}")
print("-" * 46)

for N in range(3, 11):
    xi_general = sphere_stability_threshold(N)
    
    if N % 2 == 0:
        # Even: (8N-8-N^2) / (N^2+8N-8)
        numer = 8*N - 8 - N**2
        denom = N**2 + 8*N - 8
        xi_closed = Fraction(numer, denom) if numer > 0 else None
        parity = "even"
    else:
        # Odd: (7-N) / (N+9)
        numer = 7 - N
        denom = N + 9
        xi_closed = Fraction(numer, denom) if numer > 0 else None
        parity = "odd"
    
    match = xi_general == xi_closed
    gen_str = str(xi_general) if xi_general is not None else "None"
    cls_str = str(xi_closed) if xi_closed is not None else "None"
    print(f"{N:>3}  {parity:>6}  {cls_str:>14}  {gen_str:>10}  {str(match):>6}")
    assert match, f"N={N}: closed-form {xi_closed} != general {xi_general}"

print("\nAll closed-form expressions verified.")

## Summary

All exact rational S^2 stability thresholds have been verified:

| $N$ | $\xi_{\mathrm{crit}}$ | $\varphi_{\mathrm{crit}}$ | Physical meaning |
|-----|----------------------|--------------------------|------------------|
| 3   | 1/3                  | 60.0°                    | Triangle stable above lat 30° |
| 4   | 1/5                  | 48.2°                    | Square stable above lat 41.8° |
| 5   | 1/7                  | 41.4°                    | Pentagon stable above lat 48.6° |
| 6   | 1/19                 | 25.8°                    | Hexagon stable above lat 64.2° |
| $\geq 7$ | None            | --                       | Unstable at all colatitudes |

**Key findings:**

1. All thresholds are **exactly rational** (contrast with $\mathbb{H}^2$ where they live in quadratic extensions of $\mathbb{Q}$).
2. The pattern $\xi_{\mathrm{crit}} = 1/(2N-3)$ holds for $N = 3, 4, 5$ but **breaks at $N = 6$** because $m(N-m)/2 \neq N-2$ when $N = 6$.
3. Even and odd $N$ have distinct closed-form expressions, both verified.
4. The marginal condition $C_1(S^2, \xi_{\mathrm{crit}}) = m(N-m)/2$ is satisfied **exactly** in rational arithmetic for all $N \leq 6$.